# Speed Test — All Providers / All Models

Runs sequential inference at multiple context lengths and measures tok/s.
All calls use streaming so timing reflects actual generation speed.

In [ ]:
import asyncio
import csv
import json
import subprocess
import time
from pathlib import Path

from unified_local_llm_server import LLMProviderPool
from unified_local_llm_server.llm_logger import AsyncLLMLogger
from unified_local_llm_server.provider_registry import ProviderRegistry

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
registry = ProviderRegistry.load(ROOT / "providers.example.yaml")
server = LLMProviderPool(provider_registry=registry,timeout=20)

RESULTS_DIR = ROOT / "notebooks" / "results"
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_JSON = RESULTS_DIR / "speed_results.json"

LOG_FILE = ROOT / "test_logs" / "speed_test_logger.log"
logger = AsyncLLMLogger(LOG_FILE)

print(f"Results → {RESULTS_JSON}")
print(f"Log     → {LOG_FILE}")

def vram_used_mib() -> int:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            text=True, timeout=5,
        )
        return sum(int(x) for x in out.strip().split("\n") if x.strip())
    except Exception:
        return 0

def save_result(record: dict) -> None:
    existing = json.loads(RESULTS_JSON.read_text()) if RESULTS_JSON.exists() else []
    existing.append(record)
    RESULTS_JSON.write_text(json.dumps(existing, indent=2))

Results → /home/ubn/Documents/projects/unified_local_llm_server/notebooks/results/speed_results.json
Log     → /home/ubn/Documents/projects/unified_local_llm_server/test_logs/speed_test_logger.log


## Configuration

In [ ]:
# Context lengths to test per model
CTX_LENGTHS = [12_000, 32_000, 64_000, 80_000]

# Tokens to generate per run
GEN_TOKENS = 300

# Max model size to attempt
MAX_MODEL_GB = 20.0

# Models to skip by name fragment
SKIP_FRAGMENTS = ["120b", "120B"]

# Wall-clock timeout per inference run (load + prefill + generation).
# asyncio.wait_for fires after this many seconds regardless of server keepalives.
# The background thread continues until the socket eventually times out,
# but the loop immediately moves to the next context/model.
CALL_TIMEOUT = 300.0   # seconds

# ── Context fill + needle-in-haystack ────────────────────────────────────────
CHARS_PER_TOKEN = 4.0   # base estimate; multiplied by 1.45 to match actual ~5.8 chars/tok
FILL_MULTIPLIER = 1.45  # from measure_context.py — compensates for tokenizer density
PROMPT_OVERHEAD = 120   # system prompt + task question + buffer (tokens)

NEEDLES = [
    ("PINEAPPLE7742", 0.10),
    ("MANGO3318",     0.50),
    ("KIWI9051",      0.75),
]

# Boring, repetitive sentences — predictable token density (~4 chars/token)
_FILLER_SENTENCES = [
    "The annual rainfall in the central region averages about 482 millimeters per year.",
    "Municipal water treatment plants process approximately 35 million liters daily.",
    "Crop rotation schedules typically follow a three-year cycle of cereals and legumes.",
    "Regional transportation budgets are allocated based on population density metrics.",
    "Standardised testing protocols require calibration of instruments every 90 days.",
    "Warehouse inventory systems use barcode scanning for tracking inbound shipments.",
    "Public library cataloguing follows the Dewey Decimal Classification system.",
    "Meteorological stations record wind speed, humidity, and barometric pressure hourly.",
    "Urban planning guidelines recommend a minimum of 12 square meters of green space per resident.",
    "Quality assurance audits are conducted on a quarterly basis across all manufacturing lines.",
    "The historical archive contains over 1.2 million digitized documents from the 19th century.",
    "Soil composition analysis requires pH testing alongside nitrogen and phosphorus measurements.",
    "The regional power grid distributes energy from 14 substations across the district.",
    "Building codes mandate seismic resistance ratings for structures above three stories.",
    "Statistical sampling methods use a confidence interval of 95 percent for survey data.",
]

import random

def _build_filler(target_chars: int) -> str:
    parts, total = [], 0
    while total < target_chars:
        s = random.choice(_FILLER_SENTENCES)
        parts.append(s)
        total += len(s) + 1
    return " ".join(parts)[:target_chars]

def build_prompt(ctx: int) -> list[dict]:
    fill_tokens = max(0, ctx - GEN_TOKENS - PROMPT_OVERHEAD)
    filler = _build_filler(int(fill_tokens * CHARS_PER_TOKEN * FILL_MULTIPLIER))

    # Insert needles deepest-first so earlier positions stay valid
    for needle, frac in sorted(NEEDLES, key=lambda x: x[1], reverse=True):
        pos = filler.rfind(". ", 0, int(len(filler) * frac) + 1)
        pos = pos + 2 if pos != -1 else int(len(filler) * frac)
        filler = filler[:pos] + f"\n\nThe secret code is: {needle}\n\n" + filler[pos:]

    needle_list = ", ".join(n for n, _ in NEEDLES)
    content = (
        f"Read the following document carefully:\n\n{filler}\n\n"
        f"The document contains three hidden secret codes ({needle_list}). "
        "List every secret code you found, one per line. "
        "Then briefly describe the main topic of the document."
    )
    return [{"role": "user", "content": content}]

def should_skip(name: str, size_gb: float | None = None) -> bool:
    if any(f in name for f in SKIP_FRAGMENTS):
        return True
    if size_gb is not None and size_gb > MAX_MODEL_GB:
        return True
    return False

## Provider Status

In [ ]:
providers = server.get_providers()
for provider in providers:
    a = server.restart(provider)
    print(a)

{'service': 'llama_cpp.service', 'method': 'polkit', 'returncode': 0, 'waited_s': 1.0}
{'service': 'lm_studio_serv.service', 'method': 'polkit', 'returncode': 0, 'waited_s': 0.0}
{'service': 'ollama.service', 'method': 'polkit', 'returncode': 0, 'waited_s': 1.0}
{'service': 'unsloth_studio.service', 'method': 'polkit', 'returncode': 0, 'waited_s': 3.0}


In [ ]:
statuses = {}
for name in server.get_providers():
    s = await server.check_provider(name)
    statuses[name] = s
    print(f"  {'OK' if s['ok'] else '--'}  {name:<12}  {s['server_url']}")

  OK  llama_cpp     http://127.0.0.1:9090
  OK  lm_studio     http://127.0.0.1:1234
  OK  ollama        http://127.0.0.1:11434
  OK  unsloth       http://127.0.0.1:8899


## Run Speed Test

In [ ]:
# Restart any unreachable providers that have a systemd_service configured
for provider in server.get_providers():
    if statuses.get(provider, {}).get("ok"):
        continue
    try:
        result = server.restart(provider)
        if result.get("returncode") == 0:
            print(f"[{provider}] restarted via {result['method']}  ({result.get('waited_s', 0):.1f}s)")
            statuses[provider] = await server.check_provider(provider)
            print(f"[{provider}] now {'OK' if statuses[provider]['ok'] else 'STILL DOWN'}")
        else:
            print(f"[{provider}] restart FAILED (rc={result['returncode']}): {result.get('stderr', '')}")
    except ValueError as exc:
        print(f"[{provider}] cannot restart — {exc}")
        

In [ ]:
results = []

done_keys: set[tuple] = set()
if RESULTS_JSON.exists():
    for r in json.loads(RESULTS_JSON.read_text()):
        if "error" not in r:
            done_keys.add((r["provider"], r["model"], r["ctx"]))
if done_keys:
    print(f"Resuming — {len(done_keys)} result(s) already saved, skipping those combinations.\n")

for provider in server.get_providers():
    raw_models = server.list_downloaded_models(provider)

    model_entries = []
    for m in raw_models:
        if isinstance(m, dict):
            model_entries.append((m.get("name") or m.get("path"), m.get("total_size_gb")))

    server.unload_all_models(provider)

    for model_id, size_gb in model_entries:
        if "31" not in model_id:
            continue

        for ctx in CTX_LENGTHS:
            if (provider, model_id, ctx) in done_keys:
                print(f"[{provider}] SKIP (already done) {model_id}  ctx={ctx//1000}k")
                continue

            print(f"\n{'='*60}")
            print(f"  {provider}/{model_id}  ctx={ctx//1000}k")
            print(f"{'='*60}")

            vram_before = vram_used_mib()
            messages = build_prompt(ctx)

            try:
                llm = server.load_model(provider, model_id, context_length=ctx, logger=logger)
                t0 = time.perf_counter()
                result, usage = await asyncio.wait_for(
                    llm.call(
                        return_usage=True,
                        messages=messages,
                        temperature=0.0,
                        options={"num_predict": GEN_TOKENS},
                    ),
                    timeout=CALL_TIMEOUT,
                )
                elapsed = time.perf_counter() - t0
            except asyncio.TimeoutError:
                elapsed = time.perf_counter() - t0
                print(f"  TIMEOUT after {elapsed:.0f}s  (CALL_TIMEOUT={CALL_TIMEOUT}s)")
                record = {
                    "provider": provider, "model": model_id,
                    "ctx": ctx, "error": f"timeout >{CALL_TIMEOUT:.0f}s",
                }
                results.append(record)
                save_result(record)
                continue
            except Exception as exc:
                print(f"  ERROR: {exc}")
                record = {
                    "provider": provider, "model": model_id,
                    "ctx": ctx, "error": str(exc),
                }
                results.append(record)
                save_result(record)
                continue

            vram_after  = vram_used_mib()
            vram_delta  = vram_after - vram_before
            prompt_tok  = usage.get("prompt_tokens", 0)
            gen_tok     = usage.get("completion_tokens", 0)
            total_tok   = prompt_tok + gen_tok
            gen_tps     = round(gen_tok / elapsed, 1) if elapsed > 0 else 0
            nith_pass   = all(needle in result for needle, _ in NEEDLES)

            print(f"  Prompt tok  : {prompt_tok}")
            print(f"  Gen tok     : {gen_tok}")
            print(f"  Total tok   : {total_tok}")
            print(f"  Gen speed   : {gen_tps} tok/s")
            print(f"  Total time  : {round(elapsed, 1)} s")
            print(f"  VRAM delta  : {vram_delta} MiB")
            print(f"  NITH        : {'PASS' if nith_pass else 'FAIL'}")

            record = {
                "provider":       provider,
                "model":          model_id,
                "size_gb":        size_gb,
                "ctx":            ctx,
                "prompt_tokens":  prompt_tok,
                "gen_tokens":     gen_tok,
                "total_tokens":   total_tok,
                "gen_tps":        gen_tps,
                "elapsed_s":      round(elapsed, 1),
                "vram_delta_mib": vram_delta,
                "nith_pass":      nith_pass,
            }
            results.append(record)
            save_result(record)
            server.unload_all_models(provider)
            time.sleep(2)

print(f"\n\nDone. {len(results)} new result(s) saved to {RESULTS_JSON}")

Resuming — 97 result(s) already saved, skipping those combinations.

[llama_cpp] SKIP (already done) gemma-4-31B-it-UD-Q4_K_XL  ctx=12k
[llama_cpp] SKIP (already done) gemma-4-31B-it-UD-Q4_K_XL  ctx=32k
[llama_cpp] SKIP (already done) gemma-4-31B-it-UD-Q4_K_XL  ctx=64k

  llama_cpp/gemma-4-31B-it-UD-Q4_K_XL  ctx=80k


## Summary Table

In [ ]:
model_entries

In [ ]:
csv_file = RESULTS_JSON.with_suffix(".csv")
fields = ["provider", "model", "size_gb", "ctx",
          "prompt_tokens", "gen_tokens", "total_tokens",
          "gen_tps", "elapsed_s", "vram_delta_mib", "nith_pass", "error"]

all_records = json.loads(RESULTS_JSON.read_text()) if RESULTS_JSON.exists() else []
with csv_file.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(all_records)

print(f"CSV  → {csv_file}  ({len(all_records)} rows)")

In [ ]:
ok   = [r for r in results if "error" not in r]
errs = [r for r in results if "error" in r]

print(f"{'Provider':<12}  {'Model':<45}  {'ctx':>6}  {'P.tok':>7}  {'G.tok':>6}  {'tok/s':>7}  {'Time(s)':>8}  {'VRAM MiB':>10}")
print("-" * 118)

prev_model = None
for r in ok:
    key = (r["provider"], r["model"])
    sep = "" if key == prev_model else "\n"
    prev_model = key
    size_str = f" ({r['size_gb']:.1f}G)" if r.get("size_gb") else ""
    model_label = f"{r['model']}{size_str}"
    print(f"{sep}{r['provider']:<12}  {model_label:<45}  {r['ctx']//1000:>5}k  {r['prompt_tokens']:>7}  {r['gen_tokens']:>6}  {r['gen_tps']:>7.1f}  {r['elapsed_s']:>8.1f}  {r['vram_delta_mib']:>10}")

if errs:
    print("\nErrors:")
    for r in errs:
        print(f"  {r['provider']:<12}  {r['model']:<40}  ctx={r['ctx']//1000}k  {r['error'][:60]}")

In [ ]:
server.unload_all_models(None)